<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# Cloud Storage Showcase — Local / MinIO / Real AWS S3

End-to-end demonstration of the S3-aware writer surface. The same extractor code runs against any of three storage backends — the **only thing that changes is the path string** in `manager.config`.

**What this notebook shows:**

1. The single switch (`STORAGE_MODE`) that picks between local, MinIO, and real AWS
2. How `manager.config` paths drive every writer surface (final CSV, partials, failed-IDs, HTML report)
3. End-to-end verification — list bucket contents, read back the CSV, view the HTML report
4. Cleanup of the test prefix

**Canonical reference:** see `docs/13 - Cloud_storage_principles_and_usage.md` for the design principles, the three recipes, and the common gotchas. This notebook is a runnable companion to that doc.

## Step 0: Bootstrap

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

## Step 1: Pick a storage mode

Change one variable and re-run the cells below. The downstream extractor code is identical across modes.

| Mode | When to use | Prerequisites |
|---|---|---|
| `local` | Default — outputs land in `<project-root>/results/` | None |
| `minio` | Local dev against an S3-compatible store | `docker compose -f tests/smoke_test/minio-compose.yml up -d` |
| `aws_s3` | Real AWS — for shared/team workflows | `pip install -e ".[s3]"`, `EDAGRO_S3_BUCKET` + `AWS_*` in `src/.env` |

In [ ]:
# === Pick one ===
STORAGE_MODE = "local"   # "local" | "minio" | "aws_s3"

# Optional: override the bucket without editing src/.env.
#   For aws_s3:  e.g. "my-team-bucket"
#   For minio:   e.g. "earthdaily-agriculture-dev"
BUCKET_OVERRIDE = None

## Step 2: Configure the WorkflowManager

In [ ]:
import os, uuid
from earthdaily.agriculture.services.workflow_manager import WorkflowManager

# All three modes write under a UUID-scoped prefix so this notebook is
# repeatable and easy to clean up afterwards.
run_id = uuid.uuid4().hex[:8]

if STORAGE_MODE == "local":
    from pathlib import Path as _P
    # Resolve project root via the package's notebook bootstrap (no manager yet).
    _root = _P().resolve().parent
    base = f"{_root}/results/s3_showcase/{run_id}"
    print(f"Local mode — outputs land under {base}")

elif STORAGE_MODE == "minio":
    import s3fs
    os.environ["AWS_ACCESS_KEY_ID"]     = "minioadmin"
    os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"
    os.environ["AWS_ENDPOINT_URL"]      = "http://localhost:9000"
    os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"
    bucket = BUCKET_OVERRIDE or "earthdaily-agriculture-dev"
    fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "http://localhost:9000"})
    if not fs.exists(bucket):
        fs.mkdir(bucket)  # MinIO does not auto-create buckets
    base = f"s3://{bucket}/s3_showcase/{run_id}"
    print(f"MinIO mode — outputs land under {base}")

elif STORAGE_MODE == "aws_s3":
    bucket = BUCKET_OVERRIDE or os.environ.get("EDAGRO_S3_BUCKET")
    if not bucket:
        raise RuntimeError("Set EDAGRO_S3_BUCKET in src/.env or BUCKET_OVERRIDE above.")
    if os.environ.get("AWS_ENDPOINT_URL"):
        raise RuntimeError(
            "AWS_ENDPOINT_URL is set (likely from a prior MinIO run). "
            "Restart the kernel and re-run to clear it."
        )
    base = f"s3://{bucket}/s3_showcase/{run_id}"
    print(f"Real AWS mode — outputs land under {base}")

else:
    raise ValueError(f"Unknown STORAGE_MODE={STORAGE_MODE!r}")

# Build the manager with the resolved paths in one shot — no post-construction
# patching of manager.config[...] needed. This is the doc 13 §1 pattern.
manager = WorkflowManager(
    "prod",
    log_to_console=False,
    log_level="WARNING",
    output_result_dir=f"{base}/results",
    partial_result_dir=f"{base}/partials",
)

print()
print("Effective paths:")
print(f"  output_result_dir  = {manager.output_result_dir}")
print(f"  partial_result_dir = {manager.partial_result_dir}")


## Step 3: Load a few entities

Three synthetic field polygons in central Brazil — enough to trigger partial writes and exercise the full writer surface without needing a real platform query.

In [ ]:
import pandas as pd

entities = pd.DataFrame([
    {"id": "showcase_ent_001", "geometry": "POLYGON((-51.0 -15.0, -50.9 -15.0, -50.9 -14.9, -51.0 -14.9, -51.0 -15.0))"},
    {"id": "showcase_ent_002", "geometry": "POLYGON((-50.5 -15.5, -50.4 -15.5, -50.4 -15.4, -50.5 -15.4, -50.5 -15.5))"},
    {"id": "showcase_ent_003", "geometry": "POLYGON((-49.8 -16.0, -49.7 -16.0, -49.7 -15.9, -49.8 -15.9, -49.8 -16.0))"},
])
entities

## Step 4: Run a small extraction

`CoverageExtractor` is one of the fastest extractors in the package — good for a writer-surface demo. It queries available satellite imagery for each polygon.

In [ ]:
from earthdaily.agriculture.extractors.coverage_function import CoverageExtractor

extractor = CoverageExtractor(
    manager.bearer_token,
    manager.token_expiration,
    config=manager.config,
)

extractor.setup_coverage_parameters(
    vegetation_index="NDVI",
    start_date="2025-01-01",
    end_date="2025-02-28",
    clear_cover_min=95,
)

result = extractor.process_entity_coverage_bulk_parallel(
    entity_list=entities,
    max_workers=3,
    partial_frequency=2,        # flush partials every 2 entities
    prefix="s3_showcase",
    generate_report=True,       # write an HTML report at the end
)

print()
print(f"Successful : {result['successful_calculations']}/{result['total_calculations']}")
print(f"Failed     : {result['failed_calculations']}")
print(f"Output rows: {len(result['results_df'])}")

## Step 5: Verify outputs landed in the right place

Walk the output prefix — local globs for local mode, fsspec for S3 modes. Same pattern, regardless of backend.

In [ ]:
from earthdaily.agriculture.core._fs import is_remote_path, glob_files

output_dir = manager.config["output_result_dir"]
partial_dir = manager.config["partial_result_dir"]

def _list(prefix):
    """List everything under a prefix, local or S3."""
    if is_remote_path(prefix):
        import fsspec
        fs, _ = fsspec.core.url_to_fs(prefix)
        try:
            fs.invalidate_cache()
        except Exception:
            pass
        scheme = prefix.split("://", 1)[0]
        return sorted(f"{scheme}://{k}" for k in fs.find(prefix))
    return sorted(glob_files(f"{prefix}/*"))

print("=== Results prefix ===")
for f in _list(output_dir):
    print(f"  {f}")

print()
print("=== Partials prefix (should be empty after a successful run) ===")
parts = _list(partial_dir)
for f in parts:
    print(f"  {f}")
if not parts:
    print("  (empty — cleanup pass ran)")

## Step 6: Read back the final results CSV

`pandas.read_csv` handles both local paths and `s3://` URIs transparently when given `storage_options={}`. Same line of code; the writer's storage mode never leaks into the reader.

In [ ]:
import pandas as pd
from earthdaily.agriculture.core._fs import is_remote_path

# Pick the most-recent final CSV from the results prefix.
final_csvs = sorted([f for f in _list(output_dir) if f.endswith("_final.csv") and "_results_" in f])
assert final_csvs, "no final results CSV found"
latest = final_csvs[-1]
print(f"Reading: {latest}")

storage_options = {} if is_remote_path(latest) else None
df = pd.read_csv(latest, storage_options=storage_options)
df.head()

## Step 7: View the HTML report

The HTML report is written next to the results CSV. For S3 modes it lands at `s3://.../<prefix>_report.html`; you can read it back inline with the same `storage_options={}` pattern.

In [ ]:
from IPython.display import HTML, display
import fsspec

report_candidates = [f for f in _list(output_dir) if f.endswith("_report.html")]
if report_candidates:
    report_path = report_candidates[-1]
    print(f"Report at: {report_path}")
    storage_options = {} if is_remote_path(report_path) else None
    with fsspec.open(report_path, "r", **(storage_options or {})) as fh:
        html_body = fh.read()
    display(HTML(html_body))
else:
    print("No HTML report found (generate_report=True was passed; check the run logs).")

## Step 8: Cleanup

Remove the showcase prefix so the test bucket / local results folder stays tidy. Skip this if you want to inspect the outputs further.

In [ ]:
CLEANUP = True

if CLEANUP:
    if is_remote_path(base):
        import fsspec
        fs, _ = fsspec.core.url_to_fs(base)
        fs.rm(base.split("://", 1)[1], recursive=True)
    else:
        import shutil
        shutil.rmtree(base, ignore_errors=True)
    print(f"Cleaned up {base}")
else:
    print(f"CLEANUP=False — leaving {base} in place")

## Quick test — exercise all three persistence layers from one place

Useful as a regression check after pulling new changes to the writer layer, or as a one-shot smoke test before merging an S3-touching PR. Each call is fully self-contained — its own `WorkflowManager`, its own extractor, its own UUID-scoped prefix, its own cleanup. Run the modes individually, or run all three in sequence with the last cell.

**Skip behaviour:**
- `local` — always runs.
- `minio` — skipped cleanly if MinIO isn't reachable on `localhost:9000`.
- `aws_s3` — skipped cleanly if `EDAGRO_S3_BUCKET` isn't set in `src/.env`.

**Kernel stickiness handled inline:** the function clears `AWS_ENDPOINT_URL` and the MinIO test credentials before each `aws_s3` run, then re-loads `src/.env` so real AWS creds win.

In [ ]:
def run_showcase_mode(storage_mode: str, bucket_override: str | None = None, cleanup: bool = True) -> bool:
    """Run the full Coverage showcase against one persistence layer.

    Returns True on success, False on skip / failure.
    """
    import os, uuid, time
    import pandas as pd
    from pathlib import Path
    from earthdaily.agriculture.services.workflow_manager import WorkflowManager
    from earthdaily.agriculture.extractors.coverage_function import CoverageExtractor
    from earthdaily.agriculture.core._fs import is_remote_path, glob_files

    print()
    print("=" * 64)
    print(f"  Mode: {storage_mode}")
    print("=" * 64)
    t0 = time.time()

    # --- s3 extras guard: skip cleanly if [s3] isn't installed ----------
    if storage_mode in ("minio", "aws_s3"):
        try:
            import s3fs  # noqa: F401
            import fsspec  # noqa: F401
        except ImportError as e:
            print(f"  SKIP: {e.name!r} not installed.")
            print(f"        Install the s3 extras and restart the kernel:")
            print(f"            pip install -e \".[s3]\"")
            return False

    # --- Resolve base path and any env-var setup per mode ---------------
    if storage_mode == "aws_s3":
        # Kernel stickiness: clear MinIO test creds and re-load .env so real
        # AWS creds win.
        for v in ("AWS_ENDPOINT_URL", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
            os.environ.pop(v, None)
        try:
            from dotenv import load_dotenv
            load_dotenv("src/.env", override=True)
        except Exception:
            pass

    run_id = uuid.uuid4().hex[:6]

    if storage_mode == "local":
        # Notebook lives in docs/, so project root is one level up.
        project_root = Path().resolve().parent
        base = f"{project_root}/results/s3_showcase/{run_id}"

    elif storage_mode == "minio":
        try:
            import urllib.request
            urllib.request.urlopen("http://localhost:9000/minio/health/ready", timeout=2).read()
        except Exception as e:
            print(f"  SKIP: MinIO not reachable on localhost:9000 ({type(e).__name__})")
            return False
        import s3fs
        os.environ["AWS_ACCESS_KEY_ID"]     = "minioadmin"
        os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"
        os.environ["AWS_ENDPOINT_URL"]      = "http://localhost:9000"
        os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"
        bucket = bucket_override or "earthdaily-agriculture-dev"
        fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "http://localhost:9000"})
        if not fs.exists(bucket):
            fs.mkdir(bucket)
        base = f"s3://{bucket}/s3_showcase/{run_id}"

    elif storage_mode == "aws_s3":
        bucket = bucket_override or os.environ.get("EDAGRO_S3_BUCKET")
        if not bucket:
            print("  SKIP: set EDAGRO_S3_BUCKET in src/.env or pass bucket_override=\"...\".")
            return False
        base = f"s3://{bucket}/s3_showcase/{run_id}"

    else:
        raise ValueError(f"Unknown storage_mode={storage_mode!r}")

    # Construct manager with the resolved paths in one shot - no post-
    # construction patching of manager.config[...] needed.
    manager = WorkflowManager(
        "prod",
        log_to_console=False,
        log_level="WARNING",
        output_result_dir=f"{base}/results",
        partial_result_dir=f"{base}/partials",
    )
    print(f"  base: {base}")

    # --- Run a small Coverage extraction --------------------------------
    entities = pd.DataFrame([
        {"id": "showcase_ent_001", "geometry": "POLYGON((-51.0 -15.0, -50.9 -15.0, -50.9 -14.9, -51.0 -14.9, -51.0 -15.0))"},
        {"id": "showcase_ent_002", "geometry": "POLYGON((-50.5 -15.5, -50.4 -15.5, -50.4 -15.4, -50.5 -15.4, -50.5 -15.5))"},
        {"id": "showcase_ent_003", "geometry": "POLYGON((-49.8 -16.0, -49.7 -16.0, -49.7 -15.9, -49.8 -15.9, -49.8 -16.0))"},
    ])
    extractor = CoverageExtractor(manager.bearer_token, manager.token_expiration, config=manager.config)
    extractor.setup_coverage_parameters(
        vegetation_index="NDVI", start_date="2025-01-01", end_date="2025-02-28", clear_cover_min=95,
    )
    result = extractor.process_entity_coverage_bulk_parallel(
        entity_list=entities, max_workers=3, partial_frequency=2,
        prefix="s3_showcase", generate_report=True,
    )

    # --- Verify outputs landed on the target backend --------------------
    def _list(prefix):
        if is_remote_path(prefix):
            import fsspec
            fs, _ = fsspec.core.url_to_fs(prefix)
            try: fs.invalidate_cache()
            except Exception: pass
            scheme = prefix.split("://", 1)[0]
            return sorted(f"{scheme}://{k}" for k in fs.find(prefix))
        return sorted(glob_files(f"{prefix}/*"))

    output_files = _list(manager.output_result_dir)
    final_csvs   = [f for f in output_files if "_results_" in f and f.endswith("_final.csv")]
    reports      = [f for f in output_files if f.endswith("_report.html")]
    partials_left = _list(manager.partial_result_dir)

    ok = bool(final_csvs) and bool(reports) and not partials_left
    print(f"  result        : {result['successful_calculations']}/{result['total_calculations']} successful, {result['failed_calculations']} failed")
    print(f"  output files  : {len(output_files)} (final csv={len(final_csvs)}, report={len(reports)})")
    print(f"  partials left : {len(partials_left)} (should be 0)")
    print(f"  status        : {'PASS' if ok else 'FAIL'}   ({time.time() - t0:.1f}s)")

    # --- Cleanup --------------------------------------------------------
    if cleanup:
        try:
            if is_remote_path(base):
                import fsspec
                fs, _ = fsspec.core.url_to_fs(base)
                fs.rm(base.split("://", 1)[1], recursive=True)
            else:
                import shutil
                shutil.rmtree(base, ignore_errors=True)
            print(f"  cleaned up    : {base}")
        except Exception as e:
            print(f"  cleanup failed: {e}")

    return ok


### MinIO setup sanity check

Run this **before** Test 2 (MinIO) to confirm the test fixture is healthy. Five checks: health endpoint reachable → `s3fs` installed → authentication works → test bucket exists or can be created → tiny write/read/delete round-trip. Each step has a clear remediation hint if it fails.

In [ ]:
def check_minio_setup() -> bool:
    """Quick sanity check of the MinIO test setup. Returns True if all five checks pass."""
    import urllib.request

    print("=" * 64)
    print("  MinIO setup sanity check")
    print("=" * 64)

    # 1. Health endpoint reachable
    try:
        with urllib.request.urlopen("http://localhost:9000/minio/health/ready", timeout=2) as r:
            print(f"  [OK]   MinIO health endpoint reachable (HTTP {r.status})")
    except Exception as e:
        print(f"  [FAIL] MinIO not reachable on localhost:9000")
        print(f"         {type(e).__name__}: {e}")
        print(f"         Start MinIO: docker compose -f tests/smoke_test/minio-compose.yml up -d")
        return False

    # 2. s3fs importable
    try:
        import s3fs
        print(f"  [OK]   s3fs installed (version {s3fs.__version__})")
    except ImportError:
        print(f"  [FAIL] s3fs not installed")
        print(f"         Install: pip install -e \".[s3]\" (then restart the kernel)")
        return False

    # 3. s3fs can authenticate against MinIO
    try:
        fs = s3fs.S3FileSystem(
            key="minioadmin", secret="minioadmin",
            client_kwargs={"endpoint_url": "http://localhost:9000"},
        )
        buckets = fs.ls("")
        print(f"  [OK]   s3fs authenticates against MinIO ({len(buckets)} bucket(s) visible)")
    except Exception as e:
        print(f"  [FAIL] s3fs authentication against MinIO failed")
        print(f"         {type(e).__name__}: {e}")
        return False

    # 4. Test bucket exists / can be created
    bucket = "earthdaily-agriculture-dev"
    try:
        if not fs.exists(bucket):
            fs.mkdir(bucket)
            print(f"  [OK]   Test bucket '{bucket}' created")
        else:
            print(f"  [OK]   Test bucket '{bucket}' already exists")
    except Exception as e:
        print(f"  [FAIL] Could not create or access bucket '{bucket}'")
        print(f"         {type(e).__name__}: {e}")
        return False

    # 5. Write/read/delete round-trip
    try:
        key = f"{bucket}/_sanity_check.txt"
        with fs.open(key, "w") as f:
            f.write("sanity")
        with fs.open(key, "r") as f:
            content = f.read()
        fs.rm(key)
        assert content == "sanity", f"round-trip mismatch: {content!r}"
        print(f"  [OK]   Round-trip write/read/delete works")
    except Exception as e:
        print(f"  [FAIL] Round-trip test failed")
        print(f"         {type(e).__name__}: {e}")
        return False

    print()
    print("  All checks passed - MinIO is ready for run_showcase_mode(\"minio\").")
    return True

check_minio_setup()

### Test 1 — local

In [ ]:
run_showcase_mode("local")

### Test 2 — MinIO (requires `docker compose -f tests/smoke_test/minio-compose.yml up -d`)

In [ ]:
run_showcase_mode("minio")

### Test 3 — real AWS S3 (requires `EDAGRO_S3_BUCKET` + `AWS_*` in `src/.env`)

In [ ]:
run_showcase_mode("aws_s3")

### Run all three back-to-back

Sequence: local → MinIO → AWS. Modes that aren't set up are skipped cleanly.

In [ ]:
results = {
    "local":  run_showcase_mode("local"),
    "minio":  run_showcase_mode("minio"),
    "aws_s3": run_showcase_mode("aws_s3"),
}

print()
print("=" * 64)
print("  Summary")
print("=" * 64)
for mode, ok in results.items():
    status = "PASS" if ok else "SKIP/FAIL"
    print(f"  {mode:8s}: {status}")

## Notes & gotchas

- **The kernel is sticky.** `AWS_ENDPOINT_URL` and `AWS_ACCESS_KEY_ID` persist across cells. Switching from MinIO to real AWS in the same kernel requires a kernel restart (or explicit `del os.environ[...]`).
- **Reads need `storage_options={}`.** Writes are handled by the extractor. For `pd.read_csv` / `pd.read_parquet` etc. against `s3://`, pass `storage_options={}` so pandas routes through fsspec.
- **Cache + remote = forced off.** If you point `cache_dir` at `s3://...`, the cache subsystem disables itself with a loud warning. Atomic-rename has no S3 equivalent.
- **Logs stay local.** Loguru file sink writes to `<project-root>/logs/`. For container deploys set `EDAGRO_LOG_CONSOLE_ONLY=1` (or pass `log_to_console_only=True` to `WorkflowManager`).
- **Smoke verification.** For an automated end-to-end check against real AWS, run `tests/smoke_test/cloud_smoke_test.py`. For MinIO, run `pytest tests/test_cloud_writers_integration.py` with the four MinIO env vars set.

**Related docs:** `docs/13 - Cloud_storage_principles_and_usage.md` (principles, recipes, gotchas).